# Cluster Control Panel

Пульт управления кластером федеративного обучения.
Запускать из корня проекта (`ddl/`).

| # | Модуль | Описание |
|---|--------|----------|
| 1 | **Проверка узлов** | SSH-доступность, окружение, датасеты |
| 2 | **Датасет** | Скачивание и просмотр структуры |
| 3 | **Распределение данных** | Партиционирование и rsync на узлы |
| 4 | Запуск кластера | SuperLink + SuperNodes |
| 5 | Мониторинг | Метрики обучения в реальном времени |

## Setup
Загрузка конфигурации узлов из `deploy/nodes.conf`.

In [ ]:
import subprocess
import concurrent.futures
import sys

import pandas as pd
from IPython.display import display

sys.path.insert(0, ".")
from scripts.cluster_utils import load_nodes, setup_ssh_key, ssh_node

cfg = load_nodes()
setup_ssh_key(cfg)  # копируем ключ из Windows → WSL один раз явно

print(f"Сервер : {cfg['server']['name']} ({cfg['server']['ip']})")
print(f"Клиенты: {len(cfg['clients'])}")
for c in cfg["clients"]:
    print(f"  partition-id={c['partition_id']}: {c['name']}  внешний={c['ext_ip']}  внутренний={c['ip']}")


---
## 1. Проверка узлов

Для каждого узла проверяем:
- **SSH** — доступность по ключу
- **venv** — наличие виртуального окружения
- **flwr** — версия Flower
- **dataset** — наличие папки с данными (только клиенты)

In [ ]:
def check_node(node: dict) -> dict:
    """Проверяет готовность одного узла. Возвращает dict со статусами."""
    remote_dir = cfg["remote_dir"]

    def ssh(cmd):
        r = ssh_node(cfg, node, cmd, timeout=20)
        return r.returncode == 0, r.stdout.strip()

    row = {
        "Узел":         node["name"],
        "Внешний IP":   node["ext_ip"],
        "partition-id": node["partition_id"] if node["partition_id"] is not None else "—",
        "SSH":     "✗",
        "venv":    "—",
        "flwr":    "—",
        "dataset": "—",
    }

    ok, _ = ssh("echo pong")
    if not ok:
        return row
    row["SSH"] = "✓"

    ok, _ = ssh(f"test -f {remote_dir}/.venv/bin/activate && echo yes")
    row["venv"] = "✓" if ok else "✗"

    ok, out = ssh(f"source {remote_dir}/.venv/bin/activate 2>/dev/null && "
                  f"python -c 'import flwr; print(flwr.__version__)' 2>/dev/null")
    row["flwr"] = out if ok and out else "✗"

    if node["role"] == "client":
        ok, _ = ssh(f"test -d {remote_dir}/data && echo yes")
        row["dataset"] = "✓" if ok else "✗"

    return row


print("Проверяем узлы (параллельно)...")
with concurrent.futures.ThreadPoolExecutor() as ex:
    rows = list(ex.map(check_node, cfg["all_nodes"]))

rows.sort(key=lambda r: (r["Узел"] != "fl-server", r["Узел"]))
df = pd.DataFrame(rows).set_index("Узел")
display(df)

total = len(rows)
ready = sum(1 for r in rows if r["SSH"] == "✓" and r["venv"] == "✓" and r["flwr"] != "✗")
print(f"\nГотово узлов: {ready}/{total}")
if ready < total:
    print("Запусти deploy/deploy_all.sh чтобы настроить недостающие узлы.")


---
## 2. Датасет

Скачивает датасет (если ещё нет локально) и показывает его структуру:
сплиты, количество примеров, названия классов и их распределение.

In [ ]:
import collections
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from pathlib import Path
from datasets import load_from_disk

from scripts.download_datasets import download_and_save

# ==============================================================================
# Укажи датасет для скачивания.
#
# Формат: строка идентификатора с Hugging Face Hub (huggingface.co/datasets)
#   "cifar10"          — CIFAR-10 (32x32, 10 классов, 60k изображений)
#   "mnist"            — MNIST (28x28, цифры 0-9, 70k изображений)
#   "fashion_mnist"    — Fashion-MNIST (28x28, одежда, 70k изображений)
#   "svhn"             — Street View House Numbers
#   "food101"          — Food-101 (101 класс еды)
#   "beans"            — болезни растений (3 класса)
#   "author/dataset"   — любой публичный датасет с HF Hub
#
# Найти датасеты: https://huggingface.co/datasets?task_categories=image-classification
# ==============================================================================
DATASET = "cifar10"
DATA_DIR = Path("data/")
# ==============================================================================

local_path = DATA_DIR / DATASET

print(f"Датасет : {DATASET}")
print(f"Путь    : {local_path.resolve()}")

download_and_save(DATASET, local_path)

ds = load_from_disk(str(local_path))
train = ds["train"]

# --- Сплиты и размеры ---
print(f"\n{'Сплит':<10} {'Примеров':>10}")
print("-" * 22)
for split, data in ds.items():
    print(f"{split:<10} {len(data):>10,}")

# --- Поля ---
print(f"\nПоля: {list(train.features.keys())}")

# --- Классы ---
label_feat = train.features.get("label") or train.features.get("labels")
if label_feat and hasattr(label_feat, "names"):
    class_names = label_feat.names
    print(f"\nКлассов: {len(class_names)}")
    print(f"\n{'idx':>4}  Название")
    print("-" * 20)
    for idx, name in enumerate(class_names):
        print(f"  {idx:>2}  {name}")
else:
    class_names = [str(i) for i in sorted(set(train["label"]))]
    print(f"\nНазвания классов не найдены в метаданных, используем индексы: {class_names}")


---
## 3. Распределение данных

Разбивает тренировочный датасет на партиции и рассылает их по клиентам.

**Алгоритм** `floor + scheme`:
1. Из каждого класса берём `MIN_PER_CLASS` образцов на клиента — гарантированный минимум.
2. Оставшиеся образцы распределяем по схеме: `iid` (поровну) или `dirichlet` (пропорционально Dir(α)).

**Ячейка 3.1** — конфиг + партиционирование + визуализация  
**Ячейка 3.2** — rsync партиций на клиентов и тест-сплита на сервер

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from datasets import load_from_disk
from IPython.display import display

from scripts.partition_utils import (
    partition_dataset, save_partitions,
    partition_dir_name, load_manifest,
)
from scripts.download_datasets import download_and_save

# ==============================================================================
# Параметры разбивки — меняй здесь
# ==============================================================================
DATASET       = "cifar10"    # датасет (должен совпадать с тем, что скачан в ячейке 2)
NUM_CLIENTS   = 5            # число клиентов (должно совпадать с кластером)
SCHEME        = "dirichlet"  # "iid" или "dirichlet"
ALPHA         = 0.9          # параметр Dirichlet (меньше → сильнее non-IID; игнорируется при iid)
MIN_PER_CLASS = 30           # гарантированный минимум образцов каждого класса у каждого клиента
SEED          = 42
FORCE         = False        # True — пересчитать партиции даже если уже существуют
# ==============================================================================

DATA_DIR   = Path("data/")
local_path = DATA_DIR / DATASET
part_name  = partition_dir_name(DATASET, SCHEME, NUM_CLIENTS, SEED,
                                alpha=ALPHA, min_per_class=MIN_PER_CLASS)
out_dir    = DATA_DIR / "partitions" / part_name

print(f"Датасет      : {DATASET}")
print(f"Схема        : {SCHEME}" + (f"  (alpha={ALPHA})" if SCHEME == "dirichlet" else ""))
print(f"Клиентов     : {NUM_CLIENTS}")
print(f"Min/class    : {MIN_PER_CLASS}")
print(f"Seed         : {SEED}")
print(f"Директория   : {out_dir}")
print()

# Скачиваем датасет, если ещё нет
download_and_save(DATASET, local_path)
ds = load_from_disk(str(local_path))

# Партиционирование
if out_dir.exists() and not FORCE:
    print(f"Партиции уже существуют: {out_dir}")
    print("Загружаем manifest.json...")
    manifest = load_manifest(out_dir)
else:
    print("Разбиваем датасет...")
    partitions = partition_dataset(
        ds["train"],
        num_clients=NUM_CLIENTS,
        scheme=SCHEME,
        alpha=ALPHA,
        min_per_class=MIN_PER_CLASS,
        seed=SEED,
    )
    out_dir = save_partitions(
        ds, partitions, out_dir,
        dataset=DATASET, scheme=SCHEME, alpha=ALPHA,
        min_per_class=MIN_PER_CLASS, seed=SEED, force=FORCE,
    )
    manifest = load_manifest(out_dir)

# ── Таблица: классы × клиенты ─────────────────────────────────────────────
class_names  = manifest["class_names"]
num_classes  = manifest["num_classes"]
num_clients  = manifest["num_clients"]

counts_matrix = np.array([
    [manifest["clients"][i]["classes"].get(str(c), 0) for i in range(num_clients)]
    for c in range(num_classes)
])

table_data = {f"client_{i}": counts_matrix[:, i] for i in range(num_clients)}
df = pd.DataFrame(table_data, index=class_names)
df["Σ"] = df.sum(axis=1)
display(df)

totals = [manifest["clients"][i]["total"] for i in range(num_clients)]
print(f"\nОбразцов на клиента: min={min(totals):,}  max={max(totals):,}  avg={sum(totals)//len(totals):,}")
print(f"Всего train: {sum(totals):,}  |  test: {manifest.get('test_size') or '—'}")

if SCHEME == "dirichlet":
    # Показываем фактический non-IID "перекос" по классам:
    # для каждого класса — доля образцов у самого богатого клиента
    max_shares = counts_matrix.max(axis=1) / counts_matrix.sum(axis=1)
    print(f"\nNon-IID: доля самого богатого клиента по каждому классу (идеал IID = {1/num_clients:.2f}):")
    for name, share in zip(class_names, max_shares):
        bar = "█" * int(share * 30)
        print(f"  {name:>12}: {share:.2f}  {bar}")

# ── Heatmap ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(max(6, num_clients * 1.1), max(4, num_classes * 0.55)))
im = ax.imshow(counts_matrix, aspect="auto", cmap="Blues")
plt.colorbar(im, ax=ax, label="Образцов")
ax.set_xticks(range(num_clients))
ax.set_xticklabels([f"client_{i}" for i in range(num_clients)], rotation=30, ha="right")
ax.set_yticks(range(num_classes))
ax.set_yticklabels(class_names, fontsize=8)
ax.set_xlabel("Клиент")
ax.set_ylabel("Класс")
title = f"{DATASET} / {SCHEME}"
if SCHEME == "dirichlet":
    title += f"  α={ALPHA}  min/class={MIN_PER_CLASS}"
ax.set_title(title)

thresh = counts_matrix.max() * 0.6
for r in range(num_classes):
    for c in range(num_clients):
        ax.text(c, r, f"{counts_matrix[r, c]:,}", ha="center", va="center",
                fontsize=7, color="white" if counts_matrix[r, c] > thresh else "black")

plt.tight_layout()
plt.show()

In [ ]:
# 3.2 — Удаление полного датасета с клиентов
# Удаляем {REMOTE_DIR}/data/{DATASET}/ на каждом клиенте.
# Партиции (data/partitions/) не трогаем.
import concurrent.futures
from scripts.cluster_utils import ssh_client
# Задайте явное имя датасета
DATASET = 'mnist'

def delete_dataset_on_client(node):
    path = f"{cfg['remote_dir']}/data/{DATASET}"
    r = ssh_client(cfg, node["ip"],
                   f"rm -rf {path} && echo 'deleted' || echo 'not found'",
                   timeout=30)
    status = r.stdout.strip() if r.returncode == 0 else f"FAILED: {r.stderr.strip()}"
    return node["name"], status

print(f"Удаляем {DATASET}/ на всех клиентах (параллельно)...")
with concurrent.futures.ThreadPoolExecutor() as ex:
    results = list(ex.map(delete_dataset_on_client, cfg["clients"]))

for name, status in results:
    icon = "✓" if "FAILED" not in status else "✗"
    print(f"  {icon} {name}: {status}")

In [ ]:
# 3.2 — Rsync партиций на клиентов и тест-сплита на сервер
# Запускай после ячейки 3.1 (переменные out_dir, part_name, manifest должны быть в памяти).
from scripts.cluster_utils import ssh_client, rsync_to_client, ssh_server, rsync_to_server

# ── ЯВНОЕ имя партиции ─────────────────────────────────────────
PART_NAME = "cifar10__dirichlet__a0.9__m30__n5__s42"   # ← МЕНЯЙ ЗДЕСЬ

# fallback если забыли указать
if PART_NAME:
    part_name = PART_NAME

REMOTE_PART_DIR = f"{cfg['remote_dir']}/data/partitions/{part_name}"

print(f"Используем имя партиции: {part_name}")
print(f"Удалённая директория: {REMOTE_PART_DIR}\n")

errors = []

# ── Клиенты: каждый получает свою партицию ───────────────────────────────
for node in cfg["clients"]:
    i = node["partition_id"]
    src = str(out_dir / f"client_{i}") + "/"
    dst = f"{REMOTE_PART_DIR}/client_{i}/"

    # Создаём директорию на клиенте
    r = ssh_client(cfg, node["ip"], f"mkdir -p {REMOTE_PART_DIR}/", timeout=15)
    if r.returncode != 0:
        print(f"  ✗ {node['name']} mkdir FAILED: {r.stderr.strip()}")
        errors.append(node["name"])
        continue

    r = rsync_to_client(cfg, node["ip"], src, dst, timeout=180)
    if r.returncode == 0:
        count = manifest["clients"][i]["total"]
        print(f"  ✓ {node['name']}  partition-id={i}  ({count:,} samples)")
    else:
        print(f"  ✗ {node['name']} rsync FAILED: {r.stderr.strip()}")
        errors.append(node["name"])

# ── Сервер: тест-сплит + manifest.json ────────────────────────────────────
r = ssh_server(cfg, f"mkdir -p {REMOTE_PART_DIR}/", timeout=15)
if r.returncode != 0:
    print(f"  ✗ fl-server mkdir FAILED: {r.stderr.strip()}")
    errors.append("fl-server")
else:
    test_src = str(out_dir / "test") + "/"
    r = rsync_to_server(cfg, test_src, f"{REMOTE_PART_DIR}/test/", timeout=180)
    if r.returncode == 0:
        print(f"  ✓ fl-server  test  ({manifest.get('test_size', '?')} samples)")
    else:
        print(f"  ✗ fl-server test rsync FAILED: {r.stderr.strip()}")
        errors.append("fl-server/test")

    manifest_src = str(out_dir / "manifest.json")
    r = rsync_to_server(cfg, manifest_src, f"{REMOTE_PART_DIR}/", timeout=30)
    if r.returncode == 0:
        print(f"  ✓ fl-server  manifest.json")
    else:
        print(f"  ✗ fl-server manifest rsync FAILED: {r.stderr.strip()}")
        errors.append("fl-server/manifest")

# ── Итог ──────────────────────────────────────────────────────────────────
print()
if not errors:
    print(f"✓ Все партиции разосланы в {REMOTE_PART_DIR}")
else:
    print(f"✗ Ошибки на: {', '.join(errors)}")

In [ ]:
# 3.4 — Инвентаризация партиций на клиентах
import concurrent.futures
import json
import pandas as pd
from IPython.display import display
from scripts.cluster_utils import ssh_client

# load_from_disk через venv — Arrow memory-mapped, len() это O(1)
CHECK_CMD = r"""bash -c 'source ~/ddl/.venv/bin/activate && python3 << PYEOF
import json, os
from datasets import load_from_disk

parts_dir = os.path.expanduser("~/ddl/data/partitions")
result = {}
if os.path.isdir(parts_dir):
    for part_name in sorted(os.listdir(parts_dir)):
        part_path = os.path.join(parts_dir, part_name)
        if not os.path.isdir(part_path):
            continue
        for entry in sorted(os.listdir(part_path)):
            if not entry.startswith("client_"):
                continue
            try:
                d = load_from_disk(os.path.join(part_path, entry))
                result[part_name + "/" + entry] = len(d)
            except Exception as e:
                result[part_name + "/" + entry] = -1
print(json.dumps(result))
PYEOF'"""


def scan_client(node):
    r = ssh_client(cfg, node["ip"], CHECK_CMD, timeout=60)
    if r.returncode != 0:
        return node["name"], None
    try:
        return node["name"], json.loads(r.stdout.strip())
    except (json.JSONDecodeError, ValueError):
        return node["name"], {}


print("Сканируем клиентов (параллельно)...")
with concurrent.futures.ThreadPoolExecutor() as ex:
    scan_results = list(ex.map(scan_client, cfg["clients"]))

client_names = [n["name"] for n in cfg["clients"]]

# Собираем все уникальные имена партиций
all_partitions: set[str] = set()
for _, data in scan_results:
    if data:
        for key in data:
            all_partitions.add(key.rsplit("/", 1)[0])

if not all_partitions:
    print("Партиции не найдены ни на одном клиенте.")
else:
    table: dict[str, dict[str, str]] = {p: {} for p in sorted(all_partitions)}

    for client_name, data in scan_results:
        if data is None:
            for p in table:
                table[p][client_name] = "SSH ERR"
            continue
        for key, count in data.items():
            part_name = key.rsplit("/", 1)[0]
            table[part_name][client_name] = f"{count:,}" if count >= 0 else "load err"

    for p in table:
        for cn in client_names:
            table[p].setdefault(cn, "—")

    df = pd.DataFrame(table, index=client_names).T
    df.index.name = "Партиция"
    display(df)

    for p, row in table.items():
        nums = [int(v.replace(",", "")) for v in row.values() if v.replace(",", "").isdigit()]
        print(f"  {p}: {sum(nums):,} образцов суммарно")

In [ ]:
# 3.5 — Удаление партиции с клиентов (и сервера)
import concurrent.futures
from scripts.cluster_utils import ssh_client, ssh_server

# Укажи имя партиции для удаления (можно скопировать из таблицы выше)
DELETE_PARTITION = "cifar10__dirichlet__a0.5__m50__n5__s42"

remote_path = f"{cfg['remote_dir']}/data/partitions/{DELETE_PARTITION}"
print(f"Удаляем: {remote_path}")
print(f"  на клиентах: {[n['name'] for n in cfg['clients']]}")
print(f"  на сервере:  {cfg['server']['name']}")
print()

def delete_on_client(node):
    r = ssh_client(cfg, node["ip"],
                   f"rm -rf {remote_path} && echo deleted || echo failed",
                   timeout=30)
    ok = r.returncode == 0 and "deleted" in r.stdout
    return node["name"], "✓" if ok else f"✗ {r.stderr.strip()}"

with concurrent.futures.ThreadPoolExecutor() as ex:
    results = list(ex.map(delete_on_client, cfg["clients"]))

# Сервер (хранит test/ и manifest.json)
r = ssh_server(cfg, f"rm -rf {remote_path} && echo deleted || echo failed", timeout=30)
ok = r.returncode == 0 and "deleted" in r.stdout
results.append((cfg["server"]["name"], "✓" if ok else f"✗ {r.stderr.strip()}"))

for name, status in results:
    print(f"  {status} {name}")